[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C66_Agentic_Evaluation_Course/02_outcome_scoring/02_outcome_scoring.ipynb)

# 02 · 结果判分与部分得分（变异测试 / 判分器混淆矩阵 / 排序翻转 / hack 检测 / 聚合）

目标：把「怎么判对错」从一句话，变成一套**可以被自己测试**的工程。

本 notebook 你会亲手实现：
1. **变异测试** —— 用故意注入的 bug 量化「这组测试有多强」
2. **判分器混淆矩阵与 Rogan–Gladen 校正** —— 量出 α/β，把观测成功率还原成真实成功率
3. **部分得分的排序翻转实验** —— 构造出「二值判分 A 赢、checkpoint 判分 B 赢」，并量化它多常发生
4. **checkpoint 的单调性检查器** —— 三条硬规则的可执行版本
5. **判分器 hack 检测器** —— 六种 hack 的自动化闸门
6. **micro vs macro 聚合** —— 同一批结果，两种聚合，两个排序

> 心智模型：**判分器是这套测量系统的零点。零点偏了，多测一万次只会让你更自信地相信一个错的数字。**

## 1 · 变异测试：这组测试有多强

给一个「被测函数」注入若干微小语义改动（变异体），看测试组能杀掉几个。
杀不掉的就是测试的盲区，也就是判分器的假阳性来源。

In [ ]:
import math, itertools, json
from collections import Counter, defaultdict
import numpy as np

# 被测函数的「正确实现」
def divide(a, b):
    if b == 0:
        return None
    return a / b

# 一组弱测试：只检查了 b == 0 这一条路径
WEAK_TESTS = [lambda f: f(10, 0) is None]

# 一组强测试：正常路径 + 边界 + 符号
STRONG_TESTS = [
    lambda f: f(10, 0) is None,
    lambda f: f(10, 2) == 5,
    lambda f: f(-9, 3) == -3,
    lambda f: f(7, 2) == 3.5,
]

# 变异体：每个都是一个「故意有 bug 的实现」
MUTANTS = {
    'return_const_none': lambda a, b: None,
    'drop_zero_guard':   lambda a, b: (a / b) if b != 0 else 0,
    'flip_sign':         lambda a, b: None if b == 0 else -a / b,
    'int_division':      lambda a, b: None if b == 0 else a // b,
    'swap_args':         lambda a, b: None if a == 0 else b / a,
}

def mutation_score(tests, mutants):
    """变异分数 = 被杀死的变异体 / 变异体总数。测试跑挂或断言失败即算「杀死」。"""
    killed = []
    for name, m in mutants.items():
        alive = True
        for t in tests:
            try:
                if not t(m):
                    alive = False
                    break
            except Exception:
                alive = False
                break
        if not alive:
            killed.append(name)
    return len(killed) / len(mutants), killed

ws, wk = mutation_score(WEAK_TESTS, MUTANTS)
ss, sk = mutation_score(STRONG_TESTS, MUTANTS)
print(f'弱测试变异分数: {ws:.0%}  杀死: {wk}')
print(f'强测试变异分数: {ss:.0%}  杀死: {sk}')
assert ws < 0.5 and ss == 1.0
print('\n✅ 弱测试连「全部返回 None」这种彻底错误的实现都杀不掉——')
print('   用它当判分器，等于把大量语义错误的解判成成功（假阳性）。')

In [ ]:
# 变异分数 → 该任务该不该进主指标
def scorer_tier(mut_score):
    if mut_score > 0.9:
        return 'strong'      # 可直接用作判分器
    if mut_score >= 0.6:
        return 'medium'      # 可用，但报告里注明
    return 'weak'            # 不应进入主指标

rng = np.random.default_rng(9)
task_mut = np.clip(rng.beta(5, 2, size=200), 0, 1)      # 模拟 200 个任务的测试强度分布
tiers = Counter(scorer_tier(m) for m in task_mut)
for t in ['strong', 'medium', 'weak']:
    print(f'  {t:<8} {tiers[t]:>3} 个任务 ({tiers[t]/200:.0%})')

weak_frac = tiers['weak'] / 200
print(f'\n弱判分任务占比 {weak_frac:.0%}——这部分任务的分数是不可信的。')
assert tiers['strong'] + tiers['medium'] + tiers['weak'] == 200
print('✅ 把变异分数存成任务元数据，你就能随时做「只用强判分任务重算一遍」的敏感性分析。')

## 2 · 判分器混淆矩阵与 Rogan–Gladen 校正

用金标准集（已知正确解 + 已知错误解）量出判分器的 α（假阳率）与 β（假阴率），
再把观测成功率还原成真实成功率。

In [ ]:
def confusion(gold_labels, scorer_labels):
    """gold: 真实是否正确; scorer: 判分器判定是否正确。返回 (alpha, beta, 混淆计数)。"""
    gold = np.asarray(gold_labels, dtype=bool)
    pred = np.asarray(scorer_labels, dtype=bool)
    tp = int((gold & pred).sum())
    fn = int((gold & ~pred).sum())
    fp = int((~gold & pred).sum())
    tn = int((~gold & ~pred).sum())
    alpha = fp / (fp + tn) if (fp + tn) else 0.0     # 假阳率：错的被判成对
    beta = fn / (tp + fn) if (tp + fn) else 0.0      # 假阴率：对的被判成错
    return alpha, beta, {'tp': tp, 'fn': fn, 'fp': fp, 'tn': tn}

rng = np.random.default_rng(21)
N_GOLD = 60
gold = np.array([True] * N_GOLD + [False] * N_GOLD)
TRUE_ALPHA, TRUE_BETA = 0.12, 0.05                  # 判分器的真实缺陷（弱测试 → 高假阳）
pred = np.where(gold, rng.random(2 * N_GOLD) > TRUE_BETA, rng.random(2 * N_GOLD) < TRUE_ALPHA)

alpha, beta, cm = confusion(gold, pred)
print('混淆矩阵:', cm)
print(f'估计 α(假阳率) = {alpha:.3f} | 估计 β(假阴率) = {beta:.3f}')
assert abs(alpha - TRUE_ALPHA) < 0.10 and abs(beta - TRUE_BETA) < 0.10
print('\n✅ 120 条金标准样本就能把 α/β 定位到 ±0.1 以内——这是半天的工作量。')

In [ ]:
def rogan_gladen(p_obs, alpha, beta):
    """用判分器的 α/β 把观测成功率还原成真实成功率。可能超出 [0,1]——那是信号，不是 bug。"""
    denom = 1 - beta - alpha
    if abs(denom) < 1e-9:
        return float('nan')
    return (p_obs - alpha) / denom

print(f"{'真实成功率':>10}{'观测成功率':>12}{'相对虚高':>10}{'校正回来':>10}")
for p_true in [0.05, 0.15, 0.30, 0.50, 0.80]:
    p_obs = p_true * (1 - TRUE_BETA) + (1 - p_true) * TRUE_ALPHA
    back = rogan_gladen(p_obs, TRUE_ALPHA, TRUE_BETA)
    print(f'{p_true:>10.0%}{p_obs:>12.1%}{p_obs/p_true:>9.2f}x{back:>10.1%}')
    assert abs(back - p_true) < 1e-9

print('\n注意最上面一行：真实 5% 的能力，被一个 α=12% 的判分器测成 16.4%——虚高 3.3 倍。')
print('✅ 「越难的基准上，低分越不可信」的定量版本。难基准 + 弱判分器 = 数字几乎全是假阳性。')

# 边界情形：观测值低于 α 时，校正结果为负
edge = rogan_gladen(0.08, 0.12, 0.05)
print(f'\n观测 8% 而 α=12% → 校正后 {edge:.1%}（负值）')
assert edge < 0
print('→ 正确解读是「观测到的成功可以完全由假阳性解释，无法区分于零」，而不是截断成 0 当没事发生。')

In [ ]:
def bootstrap_ci(scores, n_boot=2000, seed=0, alpha_level=0.05):
    """成功率的自举置信区间。"""
    rng = np.random.default_rng(seed)
    s = np.asarray(scores, dtype=float)
    boots = [s[rng.integers(0, len(s), len(s))].mean() for _ in range(n_boot)]
    lo, hi = np.percentile(boots, [100 * alpha_level / 2, 100 * (1 - alpha_level / 2)])
    return float(s.mean()), float(lo), float(hi)

rng = np.random.default_rng(33)
obs_scores = (rng.random(300) < 0.336).astype(float)      # 观测成功率约 33.6%
m, lo, hi = bootstrap_ci(obs_scores, seed=1)
m_c = rogan_gladen(m, TRUE_ALPHA, TRUE_BETA)
lo_c, hi_c = rogan_gladen(lo, TRUE_ALPHA, TRUE_BETA), rogan_gladen(hi, TRUE_ALPHA, TRUE_BETA)
print(f'校正前: {m:.1%}  [{lo:.1%}, {hi:.1%}]  宽度 {hi-lo:.1%}')
print(f'校正后: {m_c:.1%}  [{lo_c:.1%}, {hi_c:.1%}]  宽度 {hi_c-lo_c:.1%}')
assert (hi_c - lo_c) > (hi - lo)
print('\n✅ 校正让区间变宽了——这是诚实的代价：')
print('   原来的窄区间是「假装判分器完美」换来的，它精确但偏。')

## 3 · 部分得分的排序翻转实验

构造两个 agent：A 常常「做到一半卡住」，B 常常「要么全对要么完全跑偏」。
二值判分与 checkpoint 判分会给出相反的排名。

In [ ]:
N_CP = 4        # 每个任务 4 个 checkpoint，第 4 个 == 二值成功

def simulate_agent(n_tasks, p_step, dropout_shape, rng):
    """返回每个任务达成的 checkpoint 数（0..N_CP）。
    dropout_shape 控制「中途卡住」的倾向：'gradual' 逐步掉队，'allornothing' 要么全过要么早死。"""
    out = []
    for _ in range(n_tasks):
        if dropout_shape == 'gradual':
            k = 0
            for _ in range(N_CP):
                if rng.random() < p_step:
                    k += 1
                else:
                    break
            out.append(k)
        else:
            out.append(N_CP if rng.random() < p_step ** N_CP else 0)
    return np.array(out)

rng = np.random.default_rng(7)
A = simulate_agent(600, 0.72, 'gradual', rng)          # 稳步推进型
B = simulate_agent(600, 0.78, 'allornothing', rng)     # 孤注一掷型

binary_A, binary_B = (A == N_CP).mean(), (B == N_CP).mean()
cp_A, cp_B = A.mean() / N_CP, B.mean() / N_CP
print(f"{'':<6}{'二值成功率':>12}{'checkpoint 达成率':>20}")
print(f"{'A':<6}{binary_A:>12.1%}{cp_A:>20.1%}")
print(f"{'B':<6}{binary_B:>12.1%}{cp_B:>20.1%}")
winner_binary = 'A' if binary_A > binary_B else 'B'
winner_cp = 'A' if cp_A > cp_B else 'B'
print(f'\n二值判分赢家: {winner_binary} | checkpoint 判分赢家: {winner_cp}')
assert winner_binary != winner_cp, '本例刻意构造成排序翻转'
print('✅ 排序翻转发生了。两个结论都不是错的——它们回答的是不同的问题：')
print('   二值问「能不能交付」，checkpoint 问「走得多远」。产品决策看前者，改进方向看后者。')

In [ ]:
def flip_rate(n_trials=300, seed=0):
    """随机生成成对 agent，统计「二值排序与 checkpoint 排序不一致」的频率。"""
    rng = np.random.default_rng(seed)
    flips = 0
    for _ in range(n_trials):
        pa, pb = rng.uniform(0.5, 0.9), rng.uniform(0.5, 0.9)
        sa = simulate_agent(200, pa, 'gradual', rng)
        sb = simulate_agent(200, pb, 'allornothing', rng)
        ba, bb = (sa == N_CP).mean(), (sb == N_CP).mean()
        ca, cb = sa.mean(), sb.mean()
        if (ba > bb) != (ca > cb):
            flips += 1
    return flips / n_trials

rate = flip_rate(seed=4)
print(f'随机成对比较中，二值 vs checkpoint 排序不一致的比例: {rate:.1%}')
assert rate > 0.05, '在风格差异明显的 agent 之间，翻转绝非罕见'
print('✅ 这不是个别构造的反例——只要两个 agent 的「失败风格」不同，翻转就有可观概率发生。')
print('   报告规范：主指标二值 + 附表 checkpoint 分布，两者都给，不要二选一。')

## 4 · checkpoint 的三条硬规则：可执行的检查器

单调 / 可自动判定 / 最后一个等价于二值成功。

In [ ]:
def check_monotone(traj_checkpoints):
    """traj_checkpoints: 每条轨迹达成的 checkpoint 布尔向量（长度 N_CP）。
    单调性要求：达成第 k 个 → 必然达成前 k-1 个。返回违规轨迹的下标。"""
    bad = []
    for i, row in enumerate(traj_checkpoints):
        seen_false = False
        for v in row:
            if not v:
                seen_false = True
            elif seen_false:
                bad.append(i)
                break
    return bad

good = [[1, 1, 1, 0], [1, 0, 0, 0], [1, 1, 1, 1], [0, 0, 0, 0]]
bad_set = [[1, 0, 1, 0], [0, 1, 1, 1]]
assert check_monotone(good) == []
assert check_monotone(bad_set) == [0, 1]
print('单调性检查：合规集合无违规，违规集合两条全部检出 ✓')

def check_last_equals_binary(traj_checkpoints, binary_scores):
    """规则三：最后一个 checkpoint 必须与二值成功完全一致。"""
    last = np.array([row[-1] for row in traj_checkpoints], dtype=bool)
    return bool(np.array_equal(last, np.asarray(binary_scores, dtype=bool)))

assert check_last_equals_binary(good, [0, 0, 1, 0]) is True
assert check_last_equals_binary(good, [0, 0, 0, 0]) is False
print('规则三检查：最后一个 checkpoint 与二值成功一致性 ✓')
print('\n✅ 三条规则都是可执行的断言——把它们写进任务集的 CI，坏 checkpoint 进不了主干。')

## 5 · 判分器 hack 检测器

六种 hack 的自动化闸门。第一条（patch 触碰测试文件）必须无条件写死。

In [ ]:
TEST_PATH_PAT = ('test_', '_test.py', '/tests/', 'conftest.py', 'pytest.ini', 'tox.ini')

def touches_tests(patch_files):
    return [f for f in patch_files if any(p in f for p in TEST_PATH_PAT)]

def hack_gate(patch_files, tests_collected_before, tests_collected_after,
              env_mutations, heldout_pass):
    """返回 (是否判 0, 触发的闸门列表)。任一闸门触发即判 0。"""
    gates = []
    if touches_tests(patch_files):
        gates.append('patch_touches_tests')
    if tests_collected_after < tests_collected_before:
        gates.append('tests_disappeared')
    if env_mutations:
        gates.append('env_tampering')
    if heldout_pass is False:
        gates.append('heldout_tests_failed')
    return (len(gates) > 0, gates)

CASES = [
    (['src/utils.py'], 42, 42, [], True,  '正常的修复'),
    (['tests/test_utils.py'], 42, 42, [], True,  '改了测试文件'),
    (['src/utils.py', 'conftest.py'], 42, 40, [], True,  '跳过了两个用例'),
    (['src/utils.py'], 42, 42, ['PYTHONPATH'], True,  '改了环境变量'),
    (['src/utils.py'], 42, 42, [], False, '过了公开测试但过不了留出测试（特判输入）'),
]
for args in CASES:
    flag, gates = hack_gate(*args[:-1])
    print(f'{args[-1]:<34} 判0={flag}  闸门={gates}')

assert hack_gate(['src/a.py'], 10, 10, [], True) == (False, [])
assert hack_gate(['tests/test_a.py'], 10, 10, [], True)[0] is True
assert 'tests_disappeared' in hack_gate(['src/a.py'], 10, 8, [], True)[1]
print('\n✅ 五种 hack 全部被拦。注意最后一条：')
print('   「留出测试」是唯一能抓「特判输入」的手段——判分测试之外必须再留一组不公开的测试。')

## 6 · micro vs macro 聚合：同一批结果，两个排序

In [ ]:
# 三个子集（仓库），任务数极不均衡
SUBSETS = {'repo_big': 800, 'repo_mid': 120, 'repo_small': 80}
rng = np.random.default_rng(13)

def make_results(per_subset_rate, rng):
    rows = []
    for sub, n in SUBSETS.items():
        r = per_subset_rate[sub]
        for _ in range(n):
            rows.append({'subset': sub, 'score': float(rng.random() < r)})
    return rows

# A 在大子集上略强，B 在两个小子集上明显强
A_rows = make_results({'repo_big': 0.65, 'repo_mid': 0.30, 'repo_small': 0.25}, rng)
B_rows = make_results({'repo_big': 0.40, 'repo_mid': 0.75, 'repo_small': 0.85}, rng)

def micro(rows):
    return float(np.mean([r['score'] for r in rows]))

def macro(rows):
    by = defaultdict(list)
    for r in rows:
        by[r['subset']].append(r['score'])
    return float(np.mean([np.mean(v) for v in by.values()]))

print(f"{'':<4}{'micro':>10}{'macro':>10}")
print(f"{'A':<4}{micro(A_rows):>10.1%}{macro(A_rows):>10.1%}")
print(f"{'B':<4}{micro(B_rows):>10.1%}{macro(B_rows):>10.1%}")
assert (micro(A_rows) > micro(B_rows)) != (macro(A_rows) > macro(B_rows))
print('\n✅ micro 与 macro 给出相反的排序。这不是需要「选一个对的」的问题——')
print('   而是必须在报告里显式写出来的发现：谁更强，取决于你关心哪个子集。')

by_sub = defaultdict(dict)
for name, rows in [('A', A_rows), ('B', B_rows)]:
    for sub in SUBSETS:
        vals = [r['score'] for r in rows if r['subset'] == sub]
        by_sub[sub][name] = np.mean(vals)
print('\n按子集分解（这张表才是报告里真正有信息量的部分）:')
for sub, d in by_sub.items():
    print(f"  {sub:<12} n={SUBSETS[sub]:<4} A={d['A']:.0%}  B={d['B']:.0%}")

## ✏️ 练习 1：给定 α/β 反推「至少要多真才能被看见」

实现 `min_detectable_true_rate(alpha, beta, obs_threshold)`：
给定判分器的 α/β 和一个观测成功率阈值 `obs_threshold`，
求真实成功率至少要多少，观测值才能达到该阈值。即由
$\hat{p}_{obs} = p(1-\beta) + (1-p)\alpha$ 反解 $p$，并把结果夹到 $[0, 1]$。

In [ ]:
def min_detectable_true_rate(alpha, beta, obs_threshold):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(min_detectable_true_rate(0.0, 0.0, 0.30) - 0.30) < 1e-12
p = min_detectable_true_rate(0.12, 0.05, 0.30)
assert abs(0.12 + p * (1 - 0.05 - 0.12) - 0.30) < 1e-9
assert min_detectable_true_rate(0.40, 0.05, 0.30) == 0.0     # 观测阈值低于 α，夹到 0
assert min_detectable_true_rate(0.0, 0.5, 0.90) == 1.0       # 需要的真实率超过 1，夹到 1
print(f'α=12%,β=5% 时，要让观测值达到 30%，真实成功率只需 {p:.1%}')
print('✅ 练习 1 通过：判分器的假阳性会「送分」——观测阈值必须按 α 上移，否则门禁形同虚设。')

## ✏️ 练习 2：加权 checkpoint 得分与权重敏感性

实现 `weighted_checkpoint(cp_matrix, weights)`：`cp_matrix` 每行是一条轨迹的
checkpoint 布尔向量，返回加权平均得分（权重归一化）。
再实现 `weight_sensitivity(cp_A, cp_B, weights, delta=0.2, seed=0, n=200)`：
对权重做 ±delta 的随机扰动 n 次，返回「A 胜出的比例」。

In [ ]:
def weighted_checkpoint(cp_matrix, weights):
    # TODO
    raise NotImplementedError

def weight_sensitivity(cp_A, cp_B, weights, delta=0.2, seed=0, n=200):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
cpA = np.array([[1, 1, 0, 0], [1, 1, 1, 0], [1, 0, 0, 0]])
cpB = np.array([[1, 1, 1, 1], [1, 0, 0, 0], [0, 0, 0, 0]])
w = np.array([1.0, 1.0, 1.0, 1.0])
assert abs(weighted_checkpoint(cpA, w) - cpA.mean()) < 1e-12
w2 = np.array([4.0, 1.0, 1.0, 1.0])
assert weighted_checkpoint(cpA, w2) > weighted_checkpoint(cpA, w)   # 前置 checkpoint 加权 → A 得分上升

frac = weight_sensitivity(cpA, cpB, w, delta=0.2, seed=1, n=300)
print(f'等权时 A={weighted_checkpoint(cpA, w):.3f} B={weighted_checkpoint(cpB, w):.3f}')
print(f'权重扰动 ±20% 后，A 胜出的比例: {frac:.1%}')
assert 0.0 <= frac <= 1.0
print('✅ 练习 2 通过：胜负比例接近 50% 说明结论完全由权重决定，不该写成「A 更强」。')

## ✏️ 练习 3：hack 闸门的假阳性代价

实现 `gate_cost(n_total, n_hacks, gate_recall, gate_fpr)`：
返回 `(拦住的 hack 数, 误伤的正常解数, 净收益)`，净收益 =
拦住的 hack 数 × 10 − 误伤数 × 1（拦住一个 hack 的价值是误伤一个正常解的 10 倍）。
`n_total` 含 `n_hacks` 个作弊解。

In [ ]:
def gate_cost(n_total, n_hacks, gate_recall, gate_fpr):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
caught, hurt, net = gate_cost(1000, 50, gate_recall=1.0, gate_fpr=0.0)
assert (caught, hurt) == (50.0, 0.0) and net == 500.0
c2, h2, n2 = gate_cost(1000, 50, gate_recall=0.9, gate_fpr=0.02)
assert abs(c2 - 45.0) < 1e-9 and abs(h2 - 19.0) < 1e-9
print(f'完美闸门（patch 触碰测试文件）: 拦住 {caught:.0f} 误伤 {hurt:.0f} 净收益 {net:.0f}')
print(f'启发式闸门（recall 90%, fpr 2%）: 拦住 {c2:.0f} 误伤 {h2:.0f} 净收益 {n2:.0f}')
assert net > n2
print('✅ 练习 3 通过：零假阳性的硬规则永远优先于高召回的启发式——')
print('   这就是「patch 触碰测试文件即判 0」必须写死、而其余闸门要谨慎调阈值的原因。')

## ✏️ 练习 4：按判分强度加权的稳健成功率

实现 `robust_success_rate(scores, mut_scores, min_tier=0.6)`：
只统计变异分数 ≥ `min_tier` 的任务的成功率，并返回
`(稳健成功率, 被剔除的任务比例, 与朴素成功率之差)`。

In [ ]:
def robust_success_rate(scores, mut_scores, min_tier=0.6):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rng = np.random.default_rng(77)
n = 400
mut = np.clip(rng.beta(4, 3, size=n), 0, 1)
# 弱判分任务上成功率虚高（假阳性多）
p = np.where(mut < 0.6, 0.65, 0.35)
sc = (rng.random(n) < p).astype(float)

rob, dropped, diff = robust_success_rate(sc, mut, min_tier=0.6)
naive = sc.mean()
print(f'朴素成功率 {naive:.1%} | 稳健成功率 {rob:.1%} | 剔除 {dropped:.0%} 的任务 | 差 {diff:+.1%}')
assert rob < naive, '剔除弱判分任务后成功率应下降'
assert 0 < dropped < 1
assert abs(diff - (rob - naive)) < 1e-12
print('✅ 练习 4 通过：两个数字之差就是「弱判分任务贡献的虚高」——')
print('   这一行数字放进报告，比任何关于判分器质量的定性描述都有说服力。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def min_detectable_true_rate(alpha, beta, obs_threshold):
    denom = 1 - beta - alpha
    if abs(denom) < 1e-12:
        return float('nan')
    p = (obs_threshold - alpha) / denom
    return float(min(1.0, max(0.0, p)))

In [ ]:
# 练习 2 参考答案
def weighted_checkpoint(cp_matrix, weights):
    m = np.asarray(cp_matrix, dtype=float)
    w = np.asarray(weights, dtype=float)
    w = w / w.sum()
    return float((m * w).sum(axis=1).mean())

def weight_sensitivity(cp_A, cp_B, weights, delta=0.2, seed=0, n=200):
    rng = np.random.default_rng(seed)
    w = np.asarray(weights, dtype=float)
    wins = 0
    for _ in range(n):
        pert = w * (1 + rng.uniform(-delta, delta, size=w.shape))
        pert = np.clip(pert, 1e-9, None)
        if weighted_checkpoint(cp_A, pert) > weighted_checkpoint(cp_B, pert):
            wins += 1
    return wins / n

In [ ]:
# 练习 3 参考答案
def gate_cost(n_total, n_hacks, gate_recall, gate_fpr):
    caught = n_hacks * gate_recall
    hurt = (n_total - n_hacks) * gate_fpr
    return (caught, hurt, caught * 10 - hurt * 1)

In [ ]:
# 练习 4 参考答案
def robust_success_rate(scores, mut_scores, min_tier=0.6):
    s = np.asarray(scores, dtype=float)
    m = np.asarray(mut_scores, dtype=float)
    keep = m >= min_tier
    if keep.sum() == 0:
        return (float('nan'), 1.0, float('nan'))
    rob = float(s[keep].mean())
    dropped = float(1 - keep.mean())
    return (rob, dropped, rob - float(s.mean()))

---
## 🧪 真实工程胶囊：判分器工程的可复制骨架

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 代码类任务的判分流水线（顺序不可调换）
# ══════════════════════════════════════════════════════════════════
def score_patch(patch, task, workdir):
    # 闸门 1（硬规则，零假阳性，永远第一个跑）：patch 不许碰测试文件
    if any(p in f for f in changed_files(patch) for p in TEST_PATH_PAT):
        return {"score": 0, "reason": "patch_touches_tests"}

    # 闸门 2：在**干净的新容器**里应用 patch，不复用 agent 的运行时环境
    ctr = start_container(task["image_digest"])       # digest，不是 tag
    apply_patch(ctr, patch)

    # 闸门 3：测试收集数量不许变少（防 skip / 防改 conftest）
    n_after = collect_count(ctr, task["test_paths"])
    if n_after < task["n_tests_expected"]:
        return {"score": 0, "reason": "tests_disappeared"}

    # 主判分：F2P 必须全绿，P2P 必须保持绿
    f2p = run_tests(ctr, task["FAIL_TO_PASS"])
    p2p = run_tests(ctr, task["PASS_TO_PASS"])
    ok = f2p.all_pass and p2p.all_pass

    # 闸门 4（可选但强烈推荐）：留出测试，抓「特判输入」
    heldout = run_tests(ctr, task.get("HELDOUT_TESTS", []))
    return {"score": int(ok and heldout.all_pass),
            "f2p": f2p.summary, "p2p": p2p.summary, "heldout": heldout.summary}

# ══════════════════════════════════════════════════════════════════
# B. 判分器体检（每次任务集变更后跑一次，半天工作量）
# ══════════════════════════════════════════════════════════════════
# 1. 取 30-50 条 gold patch  → 判分器判 0 的就是假阴 → 估计 β
# 2. 取 30-50 条「破坏过的 gold patch」→ 判分器判 1 的就是假阳 → 估计 α
#    破坏方式：改一个比较运算符 / 改一个常量 / 删掉一个分支
# 3. 取 10-20 条语义等价重写 → 判分器判 0 说明它对形式差异过敏
# 4. 把 (α, β) 写进评测报告，并对主指标给出 Rogan-Gladen 校正值

# ══════════════════════════════════════════════════════════════════
# C. 变异测试（任务集构建期跑一次，结果存成任务元数据）
# ══════════════════════════════════════════════════════════════════
# pip install mutmut       # 或 cosmic-ray
# mutmut run --paths-to-mutate src/ --tests-dir tests/
# mutmut results           # 存活的变异体 = 判分盲区
# 把 mutation_score 写进 task meta，之后可做「只用 strong 任务重算」的敏感性分析。

# ══════════════════════════════════════════════════════════════════
# D. 报告模板（照抄这四行，你的报告就超过大多数公开报告）
# ══════════════════════════════════════════════════════════════════
# resolve_rate (micro):        34.2%  [31.0%, 37.5%]   n=500
# resolve_rate (macro by repo):29.8%                   12 repos
# scorer alpha / beta:         0.041 / 0.018           gold set n=120
# corrected resolve_rate:      31.5%  [27.9%, 35.2%]
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 判分器偏差不随样本量消失 | bias 是 0.10 时，把 var 再压一半毫无意义 | 先量 α/β 再扩规模 |
| 变异测试 | 测试强度决定判分强度；弱判分任务不该进主指标 | 任务集构建期 |
| 不变量优于快照 | 终态匹配升级成属性断言，天然处理等价解 | 终态类任务 |
| 部分得分会翻转排序 | 二值问「能不能交付」，checkpoint 问「走得多远」 | 主表二值 + 附表 checkpoint |
| 六种 hack | patch 触碰测试文件必须无条件判 0 | 判分流水线第一行 |
| micro vs macro | 两者排序不一致时，这件事本身就是最重要的发现 | 报告规范 |

下一模块：**03 · 轨迹级评测**——两个成功率相同的 agent，钱花在哪、卡在哪、能不能从错误里爬出来。